In [2]:
# customer_crud_test_highlight.py
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
import time

# --- 🧩 Setup Chrome Driver ---
chrome_options = Options()
chrome_options.add_argument("--start-maximized")
chrome_options.add_experimental_option("detach", True)
prefs = {"credentials_enable_service": False, "profile.password_manager_enabled": False}
chrome_options.add_experimental_option("prefs", prefs)

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()), options=chrome_options
)
wait = WebDriverWait(driver, 15)


def highlight_row(driver, text, color1="#a0e7e5", color2="#00b4d8"):
    """Make a table row blink with two alternating colors."""
    script = f"""
    const rows = document.querySelectorAll('table tr');
    rows.forEach(row => {{
        if (row.innerText.includes('{text}')) {{
            row.scrollIntoView({{behavior: 'smooth', block: 'center'}});
            row.style.transition = 'all 0.5s ease';
            let i = 0;
            const blink = setInterval(() => {{
                row.style.backgroundColor = i % 2 === 0 ? '{color1}' : '{color2}';
                i++;
                if (i > 5) clearInterval(blink);
            }}, 300);
        }}
    }});
    """
    driver.execute_script(script)


try:
    # --- 1️⃣ Login ---
    driver.get("http://localhost:3000/auth")

    email_input = wait.until(
        EC.presence_of_element_located((By.XPATH, "//input[@type='email']"))
    )
    password_input = driver.find_element(By.XPATH, "//input[@type='password']")

    email_input.send_keys("sakib2333@gmail.com")
    password_input.send_keys("Password123")

    sign_in_button = driver.find_element(
        By.XPATH, "//button[contains(text(), 'Sign In')]"
    )
    sign_in_button.click()

    wait.until(EC.url_contains("/dashboard"))
    print("✅ Logged in successfully!")

    # --- 2️⃣ Go to Customer Page ---
    driver.get("http://localhost:3000/customer")
    print("📄 Navigated to Customer page")

    # --- 3️⃣ Add Customer ---
    add_button = wait.until(
        EC.element_to_be_clickable(
            (By.XPATH, "//button[contains(text(), '+ Add Customer')]")
        )
    )
    add_button.click()

    wait.until(EC.visibility_of_element_located((By.XPATH, "//form")))
    driver.find_element(By.XPATH, "//input[@placeholder='Company Name']").send_keys(
        "Test Company LLC"
    )
    driver.find_element(By.XPATH, "//input[@placeholder='Contact Person']").send_keys(
        "John Doe"
    )
    driver.find_element(By.XPATH, "//input[@placeholder='Email']").send_keys(
        "johndoe@test.com"
    )
    driver.find_element(By.XPATH, "//input[@placeholder='Phone']").send_keys(
        "0123456789"
    )
    driver.find_element(By.XPATH, "//textarea[@placeholder='Address']").send_keys(
        "123 Test Street, Dhaka"
    )

    status_select = driver.find_element(By.XPATH, "//select[@name='status']")
    driver.execute_script(
        "arguments[0].value = arguments[1]; arguments[0].dispatchEvent(new Event('change', { bubbles: true }));",
        status_select,
        "active",
    )

    driver.find_element(By.XPATH, "//button[contains(text(), 'Save Customer')]").click()
    print("✅ Added new customer")
    time.sleep(3)

    highlight_row(driver, "Test Company LLC")
    print("🌟 Highlighted newly added customer!")
    time.sleep(3)

    # --- 4️⃣ View Customer ---
    view_button = wait.until(
        EC.element_to_be_clickable((By.XPATH, "//button[contains(text(), 'View')]"))
    )
    view_button.click()
    print("👀 Viewing customer details...")
    highlight_row(driver, "Test Company LLC", "#fff3cd", "#ffeeba")
    time.sleep(3)

    close_button = driver.find_element(By.XPATH, "//button[contains(text(), '✖')]")
    close_button.click()
    print("❌ Closed view modal")
    time.sleep(1)

    # --- 5️⃣ Edit Customer ---
    edit_button = wait.until(
        EC.element_to_be_clickable((By.XPATH, "//button[contains(text(), 'Edit')]"))
    )
    edit_button.click()

    wait.until(EC.visibility_of_element_located((By.XPATH, "//form")))
    company_input = driver.find_element(
        By.XPATH, "//input[@placeholder='Company Name']"
    )
    company_input.clear()
    company_input.send_keys("Updated Company Ltd")

    driver.find_element(
        By.XPATH, "//button[contains(text(), 'Update Customer')]"
    ).click()
    print("✏️ Edited customer")

    time.sleep(3)
    highlight_row(driver, "Updated Company Ltd", "#fff0f0", "#ffcccc")
    print("🌟 Highlighted edited customer!")
    time.sleep(3)

    # --- 6️⃣ Delete Customer ---
    delete_button = wait.until(
        EC.element_to_be_clickable((By.XPATH, "//button[contains(text(), 'Delete')]"))
    )
    delete_button.click()
    print("🗑️ Clicked delete button")

    try:
        alert = driver.switch_to.alert
        alert.accept()
        print("✅ Confirmed deletion")
    except:
        print("⚠️ No browser alert found, using custom UI instead")

    time.sleep(3)
    highlight_row(driver, "Updated Company Ltd", "#f8d7da", "#f5c2c7")
    print("💀 Highlighted row before deletion fade-out")

    # --- ✅ Verify Deletion ---
    time.sleep(3)
    if "Updated Company Ltd" not in driver.page_source:
        print("✅ Customer deleted successfully!")
    else:
        print("❌ Customer still visible after delete.")

    driver.quit()

except Exception as e:
    print(f"⚠️ Error: {e}")
    driver.quit()

✅ Logged in successfully!
📄 Navigated to Customer page
✅ Added new customer
🌟 Highlighted newly added customer!
👀 Viewing customer details...
❌ Closed view modal
✏️ Edited customer
🌟 Highlighted edited customer!
🗑️ Clicked delete button
✅ Confirmed deletion
💀 Highlighted row before deletion fade-out
✅ Customer deleted successfully!


In [3]:
# search for customer
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
import time

# --- 🧩 Setup Chrome Driver ---
chrome_options = Options()
chrome_options.add_argument("--start-maximized")
chrome_options.add_experimental_option(
    "detach", True
)  # Keep browser open after script ends
prefs = {"credentials_enable_service": False, "profile.password_manager_enabled": False}
chrome_options.add_experimental_option("prefs", prefs)

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()), options=chrome_options
)

wait = WebDriverWait(driver, 15)

try:
    # --- 1️⃣ Go to Login Page ---
    driver.get("http://localhost:3000/auth")

    # --- 2️⃣ Enter Email & Password ---
    email_input = wait.until(
        EC.presence_of_element_located((By.XPATH, "//input[@type='email']"))
    )
    password_input = driver.find_element(By.XPATH, "//input[@type='password']")

    email_input.send_keys("sakib2333@gmail.com")
    password_input.send_keys("1111")

    # --- 3️⃣ Click “Sign In” button ---
    sign_in_button = driver.find_element(
        By.XPATH, "//button[contains(text(), 'Sign In')]"
    )
    sign_in_button.click()

    # --- 4️⃣ Wait until dashboard page loads ---
    wait.until(EC.url_contains("/dashboard"))
    print("✅ Logged in successfully and dashboard loaded!")

    # --- 5️⃣ Navigate to Customer Page ---
    driver.get("http://localhost:3000/customer")

    # --- 6️⃣ Wait until search input is visible ---
    search_input = wait.until(
        EC.presence_of_element_located(
            (By.XPATH, "//input[@placeholder='Search by company or contact...']")
        )
    )

    # --- 7️⃣ Enter search term ---
    search_input.clear()
    search_input.send_keys("Test Company LLC")

    # --- 8️⃣ Click the Search button ---
    search_button = driver.find_element(By.XPATH, "//button[text()='Search']")
    search_button.click()
    print("🔍 Searching for 'Test Company LLC'...")

    # --- 9️⃣ Wait 2 minutes (120 seconds) ---
    time.sleep(4)

    # --- 🔟 Click the Cancel button to reset search ---
    cancel_button = driver.find_element(By.XPATH, "//button[text()='Cancel']")
    cancel_button.click()
    print("❌ Search canceled, results reset!")

finally:
    # Optional: wait 3 seconds before quitting
    time.sleep(3)
    driver.quit()

✅ Logged in successfully and dashboard loaded!
🔍 Searching for 'Test Company LLC'...
❌ Search canceled, results reset!


In [ ]:
# search for customer with highlight
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
import time

# --- 🧩 Setup Chrome Driver ---
chrome_options = Options()
chrome_options.add_argument("--start-maximized")
chrome_options.add_experimental_option("detach", True)  # Keep browser open after script ends
prefs = {"credentials_enable_service": False, "profile.password_manager_enabled": False}
chrome_options.add_experimental_option("prefs", prefs)

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()), options=chrome_options
)
wait = WebDriverWait(driver, 15)

try:
    # --- 1️⃣ Go to Login Page ---
    driver.get("http://localhost:3000/auth")

    # --- 2️⃣ Enter Email & Password ---
    email_input = wait.until(
        EC.presence_of_element_located((By.XPATH, "//input[@type='email']"))
    )
    password_input = driver.find_element(By.XPATH, "//input[@type='password']")
    email_input.send_keys("sakib2333@gmail.com")
    password_input.send_keys("1111")

    # --- 3️⃣ Click “Sign In” button ---
    driver.find_element(By.XPATH, "//button[contains(text(), 'Sign In')]").click()

    # --- 4️⃣ Wait until dashboard page loads ---
    wait.until(EC.url_contains("/dashboard"))
    print("✅ Logged in successfully and dashboard loaded!")

    # --- 5️⃣ Navigate to Customer Page ---
    driver.get("http://localhost:3000/customer")

    # --- 6️⃣ Wait until search input is visible ---
    search_input = wait.until(
        EC.presence_of_element_located(
            (By.XPATH, "//input[@placeholder='Search by company or contact...']")
        )
    )

    # --- 7️⃣ Enter search term ---
    search_input.clear()
    search_input.send_keys("Test Company LLC")

    # --- 8️⃣ Click the Search button ---
    search_button = driver.find_element(By.XPATH, "//button[text()='Search']")
    search_button.click()
    print("🔍 Searching for 'Test Company LLC'...")

    # --- 9️⃣ Highlight the first search result row ---
    result_row = wait.until(
        EC.presence_of_element_located((By.XPATH, "//tbody/tr"))
    )
    driver.execute_script(
        "arguments[0].scrollIntoView({behavior: 'smooth', block: 'center'});"
        "arguments[0].style.border='3px solid red'; arguments[0].style.backgroundColor='#fff8dc';",
        result_row,
    )
    print("✨ First search result highlighted")
    time.sleep(4)  # Observe the highlight

    # --- 🔟 Click the Cancel button to reset search ---
    cancel_button = driver.find_element(By.XPATH, "//button[text()='Cancel']")
    cancel_button.click()
    print("❌ Search canceled, results reset!")
    time.sleep(1)

    # --- Remove highlight after cancel ---
    driver.execute_script(
        "arguments[0].style.border=''; arguments[0].style.backgroundColor='';",
        result_row
    )
    print("🧹 Highlight removed")

finally:
    # Optional: wait a moment before quitting
    time.sleep(2)
    driver.quit()

In [2]:
# customer_add_highlight_test.py
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
import time

# --- 🧩 Setup Chrome Driver ---
chrome_options = Options()
chrome_options.add_argument("--start-maximized")
chrome_options.add_experimental_option("detach", True)
prefs = {"credentials_enable_service": False, "profile.password_manager_enabled": False}
chrome_options.add_experimental_option("prefs", prefs)

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()), options=chrome_options
)
wait = WebDriverWait(driver, 15)


def highlight_row(driver, text, color1="#a0e7e5", color2="#00b4d8"):
    """Make a table row blink with two alternating colors."""
    script = f"""
    const rows = document.querySelectorAll('table tr');
    rows.forEach(row => {{
        if (row.innerText.includes('{text}')) {{
            row.scrollIntoView({{behavior: 'smooth', block: 'center'}});
            row.style.transition = 'all 0.5s ease';
            let i = 0;
            const blink = setInterval(() => {{
                row.style.backgroundColor = i % 2 === 0 ? '{color1}' : '{color2}';
                i++;
                if (i > 5) clearInterval(blink);
            }}, 300);
        }}
    }});
    """
    driver.execute_script(script)


try:
    # --- 1️⃣ Login ---
    driver.get("http://localhost:3000/auth")

    email_input = wait.until(
        EC.presence_of_element_located((By.XPATH, "//input[@type='email']"))
    )
    password_input = driver.find_element(By.XPATH, "//input[@type='password']")

    email_input.send_keys("sakib2333@gmail.com")
    password_input.send_keys("Password123")

    sign_in_button = driver.find_element(
        By.XPATH, "//button[contains(text(), 'Sign In')]"
    )
    sign_in_button.click()

    wait.until(EC.url_contains("/dashboard"))
    print("✅ Logged in successfully!")

    # --- 2️⃣ Go to Customer Page ---
    driver.get("http://localhost:3000/customer")
    print("📄 Navigated to Customer page")

    # --- 3️⃣ Add Customer ---
    add_button = wait.until(
        EC.element_to_be_clickable(
            (By.XPATH, "//button[contains(text(), '+ Add Customer')]")
        )
    )
    add_button.click()

    wait.until(EC.visibility_of_element_located((By.XPATH, "//form")))

    driver.find_element(By.XPATH, "//input[@placeholder='Company Name']").send_keys(
        "Test Company LLC_3"
    )
    driver.find_element(By.XPATH, "//input[@placeholder='Contact Person']").send_keys(
        "Shadman Khan"
    )
    driver.find_element(By.XPATH, "//input[@placeholder='Email']").send_keys(
        "shadman@test.com"
    )
    driver.find_element(By.XPATH, "//input[@placeholder='Phone']").send_keys(
        "0123456789"
    )
    driver.find_element(By.XPATH, "//textarea[@placeholder='Address']").send_keys(
        "123 Test Street, Dhaka"
    )

    status_select = driver.find_element(By.XPATH, "//select[@name='status']")
    driver.execute_script(
        "arguments[0].value = arguments[1]; arguments[0].dispatchEvent(new Event('change', { bubbles: true }));",
        status_select,
        "active",
    )

    driver.find_element(By.XPATH, "//button[contains(text(), 'Save Customer')]").click()
    print("✅ Added new customer")

    # --- 🌟 Highlight Newly Added Customer ---
    time.sleep(3)
    highlight_row(driver, "Test Company LLC_3")
    print("🌟 Highlighted newly added customer!")

    time.sleep(5)
    driver.quit()

except Exception as e:
    print(f"⚠️ Error: {e}")
    driver.quit()

✅ Logged in successfully!
📄 Navigated to Customer page
✅ Added new customer
🌟 Highlighted newly added customer!
